# Cсылки


Ноутбуки https://cloud.mail.ru/public/ypbb/2PC7pdzzF


Данные на gdrive
url = "https://drive.google.com/file/d/1ncNsjwV38V9lLZc3PBLTwjNML9R901qX/view?usp=sharing"

In [7]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


# Data - Drom used cars

In [8]:
DATA_PATH = 'data/drom_used_cars.csv'
TARGET_COL = 'price_rub'

df = pd.read_csv(DATA_PATH)
print('Исходный размер датасета', df.shape)

TARGET_COL = 'price_rub'
DROP_COLS = ['listing_id', 
'source_url', 
'search_url', 
'model_rank', 
'model_id', 
'collected_at_utc',
'price_type',
'condition',
'labels',
'trim',
'year']


'''
Преобразования>>
price_rub переведем в миллионы
age = 2026 - year
transmission_type - 2 категории - автомат/механика
'''

df[TARGET_COL] = df[TARGET_COL]/1000000

df['age'] = 2026 - df['year']

transmission_dict = {'робот': 'автомат',
                     'АКПП': 'автомат',
                     'вариатор': 'автомат'
                    }
df['transmission_type'] = df['transmission'].replace(transmission_dict)

# Удаляем некорректные данные (УАЗ буханка, веста, Mercedes-Benz G Class электро и ГАЗ)
df = df[df['engine_volume_l']<6.5]

# Удаляем drop столбцы и пропуски в данных
df = df.drop(columns=DROP_COLS)
df = df.dropna()
print('Размер датасета после удаления пропущенных значений', df.shape)

# Выделяем числовые и категориальные признаки
num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [x for x in num_cols if x!= TARGET_COL]

cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print('\nЧисловые признаки', num_cols)
print('Категориальные признаки', cat_cols)


# Создадим сбалансированный датасет, чтобы посмотреть различия при наличии/отсутствии дисбаланса
df['is_automatic'] = df['transmission_type'].apply(lambda x: 1 if x=='автомат' else 0)
TARGET_COL = 'is_automatic'

class_counts = df[TARGET_COL].value_counts()
n = min(class_counts[0], class_counts[1])

df_balanced = pd.concat([
    df[df[TARGET_COL] == 0].sample(n=n, random_state=42),
    df[df[TARGET_COL] == 1].sample(n=n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced[TARGET_COL].value_counts())

Исходный размер датасета (8709, 26)
Размер датасета после удаления пропущенных значений (8560, 17)

Числовые признаки ['engine_volume_l', 'power_hp', 'mileage_km', 'is_damaged', 'age']
Категориальные признаки ['brand', 'brand_group', 'brand_country', 'model', 'fuel_type', 'transmission', 'drive', 'body_type', 'city', 'attributes_raw', 'transmission_type']
is_automatic
0    1749
1    1749
Name: count, dtype: int64


# LogisticRegression

LogisticRegression решает задачу классификации.
В бинарном случае: $y \in {0,1}$

Например:
- дефолт / нет дефолта;
- клиент уйдёт / не уйдёт;
- купит продукт / не купит;
- мошенническая транзакция / обычная;
- сделка состоится / не состоится.


Модель оценивает не просто класс, а вероятность:
$P(Y=1|X=x)$

То есть логистическая регрессия отвечает на вопрос:
«С какой вероятностью объект относится к классу 1 при данных значениях признаков?»

Математически:

$z=\beta_0+\beta_1x_1+\ldots+\beta_px_p$

и затем:

$p=P(Y=1|X) = \sigma(z) = \frac{1}{1+e^{-z}}.$



In [9]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)


def calculate_metrics(
    y_test,
    y_pred,
    y_pred_proba,
    y_train,
    y_pred_train,
    y_pred_train_proba,
):
    """Расчитывает метрики классификации для обучающей и тестовой выборок.

    Аргументы y_pred_proba и y_pred_train_proba должны содержать вероятности
    положительного класса.

    Возвращает DataFrame с метриками Accuracy, Precision, Recall, F1-Score,
    ROC AUC и LogLoss для обеих выборок.
    """
    test_metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_pred_proba),
        "LogLoss": log_loss(y_test, y_pred_proba),
    }

    train_metrics = {
        "Accuracy": accuracy_score(y_train, y_pred_train),
        "Precision": precision_score(y_train, y_pred_train),
        "Recall": recall_score(y_train, y_pred_train),
        "F1-Score": f1_score(y_train, y_pred_train),
        "ROC AUC": roc_auc_score(y_train, y_pred_train_proba),
        "LogLoss": log_loss(y_train, y_pred_train_proba),
    }

    return pd.DataFrame(
        {"Train": train_metrics, "Test": test_metrics}
    ).round(3)

In [10]:
def get_predictions(model, X_train, X_test):
    """Расчитывает предсказания и вероятности положительного класса.

    Использует обученную модель для обучающей и тестовой выборок.

    Returns:
        tuple: Предсказания и вероятности в порядке:
        (y_pred_train, y_pred_train_proba, y_pred_test, y_pred_test_proba).

    """
    y_pred_train = model.predict(X_train)
    y_pred_train_proba = model.predict_proba(X_train)[:, 1]

    y_pred_test = model.predict(X_test)
    y_pred_test_proba = model.predict_proba(X_test)[:, 1]

    return (
        y_pred_train,
        y_pred_train_proba,
        y_pred_test,
        y_pred_test_proba,
    )

In [11]:
# СОздадим сбалансированный датасет

cols = ['price_rub', 'age', 'power_hp', 'mileage_km']
df['is_automatic'] = df['transmission_type'].apply(lambda x: 1 if x=='автомат' else 0)
TARGET_COL = 'is_automatic'

X=df_balanced[cols]
y=df_balanced[TARGET_COL]
print("Доли классов\n",y.value_counts(normalize=True))

# Разделим на test/train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3 )

# 
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

Доли классов
 is_automatic
0    0.5
1    0.5
Name: proportion, dtype: float64


# Посмотрим на качество наивной модели предсказания 1 класса для всех объектов

In [ ]:
from sklearn.dummy import DummyClassifier

model = DummyClassifier(strategy='most_frequent')
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred = model.predict(X_test)


calculate_metrics(y_test, y_pred, y_pred, y_train, y_pred_train, y_pred_train)

# LogisticRegression и методы
- fit()
- predict()
- predict_proba()
- decision_function()

In [18]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(verbose=1)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

y_pred_train = model.predict(X_train_scaled)
y_pred_train_proba = model.predict_proba(X_train_scaled)

print(calculate_metrics(y_test, y_pred, y_pred, y_train, y_pred_train, y_pred_train))

print('\nDecision function ', model.decision_function(X_test_scaled)[:3])


           Train   Test
Accuracy   0.911  0.903
Precision  0.927  0.923
Recall     0.895  0.867
F1-Score   0.911  0.894
ROC AUC    0.911  0.901
LogLoss    3.208  3.501

Decision function  [-3.27114535 42.03680876 -1.7000626 ]


# Атрибуты обученной модели

In [19]:
# Атрибуты

print('Коэффициенты: ', model.coef_)
print('Свободный коэффициент: ', model.intercept_)
print('Количество итераций: ',model.n_iter_)
print('Классы: ', model.classes_)

Коэффициенты:  [[ 3.61648594 -0.91890209  4.38177484  0.27775116]]
Свободный коэффициент:  [1.88294682]
Количество итераций:  [15]
Классы:  [0 1]


# LogisticRegression умеет работать с категориальным таргетом

In [20]:
TARGET_COL = 'transmission_type'
y_cat=df_balanced[TARGET_COL]

# Разделим на test/train
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=3 )
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)


model = LogisticRegression(verbose=1)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:,1]

print('Accuracy: ', accuracy_score(y_test, y_pred))
print(model.classes_)

Accuracy:  0.9028571428571428
['автомат' 'механика']


# Параметры

In [21]:
help(LogisticRegression)

Help on class LogisticRegression in module sklearn.linear_model._logistic:

class LogisticRegression(sklearn.callback._callback_support.CallbackSupportMixin, sklearn.linear_model._base.LinearClassifierMixin, sklearn.linear_model._base.SparseCoefMixin, sklearn.base.BaseEstimator)
 |  LogisticRegression(penalty='deprecated', *, C=1.0, l1_ratio=0.0, dual=False, tol=0.0001, fit_intercept=True, intercept_scaling=1, class_weight=None, random_state=None, solver='lbfgs', max_iter=100, verbose=0, warm_start=False, n_jobs=None)
 |
 |  Logistic Regression (aka logit, MaxEnt) classifier.
 |
 |  This class implements regularized logistic regression using a set of available
 |  solvers. **Note that regularization is applied by default**. It can handle both
 |  dense and sparse input `X`. Use C-ordered arrays or CSR matrices containing 64-bit
 |  floats for optimal performance; any other input format will be converted (and
 |  copied).
 |
 |  The solvers 'lbfgs', 'newton-cg', 'newton-cholesky' and 's

LogisticRegression(
    C=1.0, 
    l1_ratio=0.0, 
    
    tol=0.0001, 
    max_iter=100,

    fit_intercept=True, 
    class_weight=None, 
    random_state=None, 
    
    solver='lbfgs',
    warm_start=False
    )
 |

In [ ]:
models = {'LR_lbfgs': LogisticRegression(solver='lbfgs'),
          'LR_saga': LogisticRegression(solver='saga'),
          'LR_liblinear': LogisticRegression(solver='liblinear'),
           }

for m_name, model in models.items():
    
    print(f'\n{m_name}')
    model.fit(X_train_scaled, y_train)
    print(model.coef_, model.intercept_)

    y_pred_train, y_pred_train_proba, y_pred_test, y_pred_test_proba =\
        get_predictions(model, X_train_scaled, X_test_scaled)
    print(calculate_metrics(y_test, y_pred_test ,y_pred_test_proba, 
                            y_train, y_pred_train, y_pred_train_proba))
    print('Iterations: ', model.n_iter_)

## Scaled vs not scaled

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split


# Данные для демонстрации — используем отдельные имена,
# чтобы не перезаписывать X и y из других частей ноутбука.
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

print('X.shape ', X.shape)#, X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Масштабируем, обучая scaler только на обучающей выборке.
scaler_demo = StandardScaler()
X_train_scaled = scaler_demo.fit_transform(X_train)
X_test_scaled = scaler_demo.transform(X_test)

# Одинаковые параметры моделей: меняется только масштаб признаков.
model_unscaled = LogisticRegression(C=1.0, max_iter=1000)
model_scaled = LogisticRegression(C=1.0, max_iter=1000)

model_unscaled.fit(X_train, y_train)
model_scaled.fit(X_train_scaled, y_train)

results = []

for name, model, X_test_model in [
    ("Без масштабирования", model_unscaled, X_test),
    ("Со StandardScaler", model_scaled, X_test_scaled),
]:
    predictions = model.predict(X_test_model)
    probabilities = model.predict_proba(X_test_model)[:, 1]

    results.append({
        "Вариант": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "ROC AUC": roc_auc_score(y_test, probabilities),
        "LogLoss": log_loss(y_test, probabilities),
        "Итераций": model.n_iter_[0],
    })

pd.DataFrame(results).set_index("Вариант").round(3)


# Solver

| Solver | Когда выбирать | Регуляризация | Особенности |
|---|---|---|---|
| `lbfgs` | универсальный default для небольших/средних dense данных | L2, none | хороший общий выбор |
| `liblinear` | маленькие датасеты, binary classification | L1, L2 | медленнее на больших данных, OVR для multiclass |
| `newton-cg` | небольшие/средние dense данные | L2, none | использует 2-й порядок, может быть точным, но тяжелее |
| `newton-cholesky` | много наблюдений, относительно мало признаков | L2, none | очень быстро при \($n \gg p$\), но память растёт как \($p^2$\) |
| `sag` | очень большие датасеты | L2, none | требует scaling |
| `saga` | большие данные, sparse/high-dimensional, L1/ElasticNet | L1, L2, ElasticNet, none | самый гибкий для regularization, требует scaling |

random_state не используется солверами lbfgs, newton-cg и newton-cholesky

**Solver** посмотрим на датасете

**Covertype** — данные о типах лесного покрова в районе национального леса Рузвельта в США.


- 581 012 объектов, 
- 54 признака: высота, уклон, расстояние до воды и дорог, тип почвы и другие характеристики.
- Целевая переменная — один из 7 типов лесного покрова.

In [ ]:
from time import perf_counter
import warnings


from sklearn.datasets import fetch_covtype
from sklearn.exceptions import ConvergenceWarning


# При первом запуске датасет будет скачан.
covtype = fetch_covtype()
X_cov, y_cov = covtype.data, covtype.target
X_sample, y_sample = X_cov, y_cov 

# # Стратифицированно берём 120 000 объектов для более быстрого эксперимента.
# X_sample, _, y_sample, _ = train_test_split(
#     X_cov,
#     y_cov,
#     train_size=120_000,
#     random_state=42,
#     stratify=y_cov,
# )

X_train_cov, X_test_cov, y_train_cov, y_test_cov = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    random_state=42,
    stratify=y_sample,
)

# Для честного сравнения все solvers получают одни и те же масштабированные данные.
scaler_cov = StandardScaler()
X_train_cov_scaled = scaler_cov.fit_transform(X_train_cov)
X_test_cov_scaled = scaler_cov.transform(X_test_cov)

results = []

for solver in ["lbfgs", "newton-cg", "sag", "saga"]:
    model = LogisticRegression(
        solver=solver,
        C=1.0,
        max_iter=100,
        tol=1e-3,
        random_state=42,
    )

    start = perf_counter()

    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X_train_cov_scaled, y_train_cov)

    elapsed = perf_counter() - start
    y_pred_cov = model.predict(X_test_cov_scaled)

    results.append({
        "Solver": solver,
        "Время, сек": round(elapsed, 2),
        "Итераций": int(model.n_iter_.max()),
        "Accuracy": round(accuracy_score(y_test_cov, y_pred_cov), 4),
        "Сходимость": (
            "есть предупреждение"
            if any(issubclass(w.category, ConvergenceWarning) for w in caught_warnings)
            else "без предупреждения"
        ),
    })

pd.DataFrame(results).set_index("Solver")

# Дисбаланс

In [ ]:
cols = ['price_rub', 'age', 'power_hp', 'mileage_km']
TARGET_COL = 'is_automatic'

X=df[cols]
y=df[TARGET_COL]
print("Доли классов\n",y.value_counts(normalize=True))

# Разделим на test/train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3 )

# 
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [ ]:
model = DummyClassifier(strategy='most_frequent')
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred = model.predict(X_test)


calculate_metrics(y_test, y_pred, y_pred, y_train, y_pred_train, y_pred_train)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred_train, y_pred_train_proba, y_pred, y_pred_proba = get_predictions(model, X_train_scaled, X_test_scaled)
calculate_metrics(y_test, y_pred, y_pred_proba, y_train, y_pred_train, y_pred_train_proba)

print('Confusion_matrix\n', confusion_matrix(y_test, y_pred))
print('\nClassification_report\n', classification_report(y_test, y_pred))

# Ограничения
- мультиколлинеарность -> нестабильные коэффициенты
- выбросы -> влияние на коэффициенты
- разделимость по признаку -> коэффициенты могут стать очень большими
- регуляризация - всегда trade of между bias/variance. Сильная -> bias, слабая -> variance/overfitting
- важно масштабирование


# Когда применять
- как быструю модель для baseline
- когда важны простота и интерпретируемость
- когда предполагаем приближенно линейную зависимость между признаками и логарифмом шансов(log odds)


# MultinomialRegression

In [ ]:
from time import perf_counter
import warnings


from sklearn.datasets import fetch_covtype
from sklearn.exceptions import ConvergenceWarning


# При первом запуске датасет будет скачан.
covtype = fetch_covtype()
X_cov, y_cov = covtype.data, covtype.target
X_sample, y_sample = X_cov, y_cov 

# Стратифицированно берём 120 000 объектов для более быстрого эксперимента.
X_sample, _, y_sample, _ = train_test_split(
    X_cov,
    y_cov,
    train_size=20_000,
    random_state=2,
    stratify=y_cov,
)

X_train_cov, X_test_cov, y_train_cov, y_test_cov = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    random_state=42,
    stratify=y_sample,
)

# Для честного сравнения все solvers получают одни и те же масштабированные данные.
scaler_cov = StandardScaler()
X_train_cov_scaled = scaler_cov.fit_transform(X_train_cov)
X_test_cov_scaled = scaler_cov.transform(X_test_cov)

results = []


model = LogisticRegression(
    solver="lbfgs",
    C=1.0,
    max_iter=1000,
    tol=1e-4,
    random_state=42,
)

start = perf_counter()
model.fit(X_train_cov_scaled, y_train_cov, )

y_pred_cov = model.predict(X_test_cov_scaled)

results.append({
    "Итераций": int(model.n_iter_.max()),
    "Accuracy": round(accuracy_score(y_test_cov, y_pred_cov), 4),
})

pd.DataFrame(results)

In [ ]:
print('Классы ', model.classes_)
print('Размерность коэффициентов', model.coef_.shape)
print('Свободный член ', model.intercept_)

In [ ]:
print(confusion_matrix(y_test_cov, y_pred_cov))

In [ ]:
from sklearn.metrics import balanced_accuracy_score, log_loss, roc_auc_score

# Вероятности для каждого класса; порядок столбцов соответствует model.classes_
y_proba_cov = model.predict_proba(X_test_cov_scaled)

balanced_acc = balanced_accuracy_score(y_test_cov, y_pred_cov)

multiclass_logloss = log_loss(
    y_test_cov,
    y_proba_cov,
    labels=model.classes_,
)

roc_auc_ovr = roc_auc_score(
    y_test_cov,
    y_proba_cov,
    labels=model.classes_,
    multi_class="ovr",
    average="weighted",
)

print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Multiclass log loss: {multiclass_logloss:.4f}")
print(f"ROC AUC OVR (macro): {roc_auc_ovr:.4f}")

In [ ]:
print(classification_report(y_test_cov, y_pred_cov))

# Решающие деревья

Линейная модель проще и надёжнее, если зависимость действительно близка к линейной.
Что же делать, если мы предполагаем, что зависимость сложнее?

Преимущество метода решающих деревьев - учет нелинейных зависимостей и взаимодействия признаков

Еще достоинства метода:
- можно визуализировать;
- не требуется масштабирование признаков;

Что необходимо знать
- деревья склонны к переобучению; 
- могут быть нестабильны при небольшом изменении данных


Давайте разберем
0. Как обучать и оценивать;
1. Как дерево принимает решение;
2. Как выбирается разбиение;
3. Как управлять сложностью и переобучением;
4. Как дерево работает с несколькими классами;



In [ ]:
n = 500
X = np.zeros(shape = (n, 2))
X[:, 0] = np.linspace(-5, 5, 500)
X[:, 1] = X[:, 0] + 0.5 * np.random.normal(size=n)
y = (X[:, 1] > X[:, 0]).astype(int)

f,ax = plt.subplots(figsize=(7,5))
ax.scatter(X[:, 0], X[:, 1], s = 10, c = y, cmap = "winter")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, 
                                                    random_state = 13)

lr = LogisticRegression(random_state = 13)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print(f"Linear model accuracy: {accuracy_score(y_pred_lr, y_test):.2f}")

In [ ]:
# !pip install mlxtend
from mlxtend.plotting import plot_decision_regions

f,ax = plt.subplots(figsize=(6,4))
plot_decision_regions(X_test, y_test, lr, ax=ax)

plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state = 13)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f"Decision tree accuracy: {accuracy_score(y_pred_dt, y_test):.2f}")

f,ax = plt.subplots(figsize=(9,6))
plot_decision_regions(X_test, y_test, dt, ax=ax)

plt.show()

In [ ]:
np.random.seed(13)
X = np.random.randn(500, 2)
y = np.logical_xor(X[:, 0] > 0, X[:, 1] > 0).astype(int)

f, ax = plt.subplots(figsize=(5,5))
plt.scatter(X[:, 0], X[:, 1], s = 10, c = y, cmap = "winter")
plt.show()


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, 
                                                    random_state = 13)

lr = LogisticRegression(random_state = 13)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

dt = DecisionTreeClassifier(random_state = 13)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)



print(f"Linear model accuracy: {accuracy_score(y_pred_lr, y_test):.2f}")
print(f"Decision tree accuracy: {accuracy_score(y_pred_dt, y_test):.2f}")


ig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))

plot_decision_regions(X_test, y_test, clf=lr, ax=ax[0])
ax[0].set_title("Logistic Regression")

plot_decision_regions(X_test, y_test, clf=dt, ax=ax[1])
ax[1].set_title("Decision Tree")

plt.tight_layout()
plt.show()


# Методы, атрибуты и параметры


## Методы


- fit(X, y) — обучить дерево;
- predict(X) — предсказать классы;
- predict_proba(X) — получить вероятности классов;
- score(X, y) — посчитать accuracy;
- apply(X) — получить номера листьев для объектов;
- decision_path(X) — получить путь объектов по узлам дерева;
- get_depth() и get_n_leaves() — глубина и число листьев обученного дерева.

In [ ]:
dt = DecisionTreeClassifier(max_depth=13, random_state = 13)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_pred_dt_proba = dt.predict_proba(X_test)
dt.score(X_test, y_test)
dt.apply(X_test)
path = dt.decision_path(X_test)
for i in range(3):  # первые 5 объектов
    start, end = path.indptr[i], path.indptr[i + 1]
    print(f"Объект {i}: узлы {path.indices[start:end]}")
dt.get_depth()
dt.get_n_leaves()

## Атрибуты

- classes_ — классы модели;
- feature_importances_ — важность признаков;
- n_features_in_ — число признаков при обучении;
- feature_names_in_ — имена признаков, если при обучении передавался DataFrame;
- tree_ — внутреннее представление дерева.

In [ ]:
dt.classes_
dt.n_features_in_

from sklearn.tree import export_text
print(export_text(dt, feature_names=["x0", "x1"]))

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize = (12,10))
plot_tree(dt, feature_names = [str(x) for x in range(2)], filled = True, rounded = True)
plt.show()

In [ ]:
pd.DataFrame({
    "feature": list(range(2)),
    "importance": dt.feature_importances_
}).sort_values(by = "importance", ascending = False).reset_index(drop = True)

## Параметры

- criterion — критерий разбиения: gini, entropy или log_loss;
- max_depth — максимальная глубина;
- min_samples_split, min_samples_leaf — минимальное число объектов для разбиения и в листе;
- max_features — число признаков, рассматриваемых при поиске разбиения;
- class_weight — веса классов;
- random_state — фиксирует случайность.



Качество разбиения - критерий 
Способ поиска разбиения - splitter
Ограничение роста дерева - max_depth, min_samples_split, min_samples_leaf, max_leaf_nodes, min_impurity_decrease
Выбор признаков - max_features
Случайность - random_state




In [ ]:
from sklearn.datasets import load_breast_cancer


# Загружаем датасет в DataFrame
cancer = load_breast_cancer(as_frame=True)
X_tree_demo = cancer.data
y_tree_demo = cancer.target

# Делим данные со сохранением долей классов
X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_tree_demo,
    y_tree_demo,
    test_size=0.2,
    random_state=42,
    stratify=y_tree_demo,
)

tree_demo = DecisionTreeClassifier(max_depth=13, random_state=42)
tree_demo.fit(X_train_tree, y_train_tree)

In [ ]:
dt.get_params()

## Критерии разбиения

Для каждого возможного разбиения дерево считает взвешенную неоднородность дочерних узлов и выбирает разбиение, которое сильнее всего её уменьшает.

- Gini: (G = 1 - $sum_k p_k^2$). Обычно используется по умолчанию; вычисляется немного проще.
- Entropy: (H = -$sum_k p_k \log_2 p_k$). Измеряет неопределённость распределения классов.
- Log loss: для оценки разбиения в дереве sklearn фактически использует тот же критерий Шеннона, что и для entropy. Названия отражают разные интерпретации: энтропия — неопределённость, log loss — ошибка вероятностного прогноза. На практике entropy и log_loss обычно дают такие же или очень похожие деревья, что и gini. У




In [ ]:
for criterion in ["gini", "entropy", "log_loss"]:
    dt = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=5,
        random_state=13,
    )
    dt.fit(X_train, y_train)
    print("Accuracy ", criterion, dt.score(X_test, y_test))

# splitter

задаёт способ поиска разбиения в каждом узле
- best — дерево рассматривает возможные разбиения и выбирает то, которое сильнее всего уменьшает неоднородность узлов.
- random — дерево случайно выбирает кандидатов на разбиение, а затем берёт лучший из них по выбранному criterion. Для воспроизведения нужен random_state 

In [ ]:
for spltr in ["best", "random"]:
    dt = DecisionTreeClassifier(
        criterion='gini',
        splitter=spltr,
        max_depth=5,
        random_state=13,
    )
    dt.fit(X_train, y_train)
    print(spltr, dt.score(X_test, y_test))

# Breast Cancer dataset

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, recall_score, f1_score



# Загружаем датасет в DataFrame
cancer = load_breast_cancer(as_frame=True)
X_tree_demo = cancer.data
y_tree_demo = cancer.target

# Делим данные со сохранением долей классов
X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_tree_demo,
    y_tree_demo,
    test_size=0.2,
    random_state=42,
    stratify=y_tree_demo,
)

res = []
for md in [3,4,5,6,7]:
# Дерево без ограничений глубины — полезно для демонстрации переобучения
    tree_demo = DecisionTreeClassifier(max_depth=md, random_state=42)
    tree_demo.fit(X_train_tree, y_train_tree)

    train_accuracy = f1_score(
        y_train_tree, tree_demo.predict(X_train_tree)
    )
    test_accuracy = f1_score(
        y_test_tree, tree_demo.predict(X_test_tree)
    )
    res.append([md, train_accuracy, test_accuracy])

res = pd.DataFrame(res, columns = ['md', 'train_accuracy', 'test_accuracy'])
f,ax = plt.subplots(figsize=(9,4))
ax.set_title('Качество от max_depth')
ax.plot(res['md'], res['train_accuracy'], label='train')
ax.plot(res['md'], res['test_accuracy'], label='test')
plt.legend()
plt.show()


In [ ]:
tree_demo = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_demo.fit(X_train_tree, y_train_tree)
train_accuracy = accuracy_score(
        y_train_tree, tree_demo.predict(X_train_tree)
    )
test_accuracy = accuracy_score(
        y_test_tree, tree_demo.predict(X_test_tree)
    )

# Кросс-валидация проводится только на обучающей выборке
cv_scores = cross_val_score(
    tree_demo,
    X_train_tree,
    y_train_tree,
    cv=5,
    scoring="accuracy",
)

print(f"Accuracy на обучающей выборке: {train_accuracy:.3f}")
print(f"Accuracy на тестовой выборке:  {test_accuracy:.3f}")
print(f"Средняя accuracy при 5-fold CV: {cv_scores.mean():.3f}")
print(f"Глубина дерева: {tree_demo.get_depth()}")
print(f"Количество листьев: {tree_demo.get_n_leaves()}")

gap = train_accuracy - test_accuracy
print(f"\nРазница train и test accuracy: {gap:.3f}")

if gap > 0.05:
    print("Есть признаки возможного переобучения: качество на train заметно выше.")
else:
    print("Заметного переобучения по разнице train/test accuracy не видно.")

# График 12 наиболее важных признаков
importance_df = (
    pd.DataFrame({
        "feature": X_tree_demo.columns,
        "importance": tree_demo.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .head(12)
)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="importance", y="feature", color="steelblue")
plt.title("Наиболее важные признаки решающего дерева")
plt.xlabel("Важность признака")
plt.ylabel("Признак")
plt.tight_layout()
plt.show()

# Критерии разбиения своими руками

In [ ]:


def gini_impurity(y):
    """Неоднородность классов в узле по критерию Gini."""
    if len(y) == 0:
        return 0.0

    probabilities = np.unique(y, return_counts=True)[1] / len(y)
    return 1 - np.sum(probabilities ** 2)


def entropy_impurity(y):
    """Неоднородность классов в узле по энтропии."""
    if len(y) == 0:
        return 0.0

    probabilities = np.unique(y, return_counts=True)[1] / len(y)
    return -np.sum(probabilities * np.log2(probabilities))


def split_node(X, y, feature, threshold):
    """Делит выборку на левую (<= threshold) и правую (> threshold) части."""
    left_mask = X[feature] <= threshold
    right_mask = ~left_mask

    return (
        X.loc[left_mask], y.loc[left_mask],
        X.loc[right_mask], y.loc[right_mask],
    )


def split_quality(X, y, feature, threshold, criterion="gini"):
    """Считает взвешенную неоднородность после разбиения. Меньше — лучше."""
    if criterion == "gini":
        impurity = gini_impurity
    elif criterion == "entropy":
        impurity = entropy_impurity
    else:
        raise ValueError("criterion должен быть 'gini' или 'entropy'")

    X_left, y_left, X_right, y_right = split_node(X, y, feature, threshold)

    if len(y_left) == 0 or len(y_right) == 0:
        return np.inf

    n = len(y)
    return (
        len(y_left) / n * impurity(y_left)
        + len(y_right) / n * impurity(y_right)
    )

In [ ]:
feature = X_train_tree.columns[0]
thresholds = np.sort(X_train_tree[feature].unique())

gini_scores = [
    split_quality(X_train_tree, y_train_tree, feature, t, criterion="gini")
    for t in thresholds
]
entropy_scores = [
    split_quality(X_train_tree, y_train_tree, feature, t, criterion="entropy")
    for t in thresholds
]

best_gini_threshold = thresholds[np.argmin(gini_scores)]
best_entropy_threshold = thresholds[np.argmin(entropy_scores)]

print("Лучший порог Gini:", best_gini_threshold)
print("Качество Gini:", min(gini_scores))
print("Лучший порог entropy:", best_entropy_threshold)
print("Качество entropy:", min(entropy_scores))

